In [36]:
Features = [
    'item_nbr',
    'family',
    'class',
    'perishable',
    'has_float_sales',
    'is_likely_closed',
    'day_of_week',
    'is_weekend',
    'is_month_start',
    'is_month_end',
    'item_mean_sales',
    'item_std_sales',
    'sales_vs_family_mean',
    'lag_1',
    'lag_3',
    'lag_5',
    'lag_7',
    'lag_14',
    'lag_30',
    'lag_60',
    'lag_140',
    'rolling_mean_3',
    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_30',
    'rolling_mean_140',
    'rolling_std_7',
    'rolling_std_14',
    'rolling_std_30',
    'rolling_max_7',
    'rolling_min_7',
    'transactions_lag_1',
    'transactions_lag_3',
    'transactions_lag_7',
    'transactions_lag_14',
    'transactions_rolling_7',
    'transactions_rolling_14',
    'transactions_rolling_30',
    'traffic_ratio',
    'oil_price_lag_1',
    'oil_price_lag_3',
    'oil_price_lag_7',
    'oil_price_rolling_mean_7',
    'oil_price_rolling_mean_30',
    'oil_price_change_7',
    'promo_lag_1',
    'promo_lag_3',
    'promo_lag_7',
    'promo_count_7',
    'promo_count_30',
    'days_since_last_promo',
    'days_since_last_holiday',
    'days_to_next_holiday',
    'Future_is_Holiday',
    'Future_is_National',
    'Future_is_Local',
    'Future_is_likely_closed',
    'Future_day_of_week',
    'Future_is_weekend',
    'Future_is_month_start',
    'Future_is_month_end',
    'Future_On_Promotion'
]
len(Features)

62

In [37]:
Target = 'target'

In [38]:
Categorical_Features = [
    'item_nbr',
    'family',
    'class'
]

In [39]:
Future_Features_Map = {
    'is_Holiday' : 'Future_is_Holiday',
    'is_National' : 'Future_is_National',
    'is_Local' : 'Future_is_Local',
    'is_likely_closed' : 'Future_is_likely_closed',
    'day_of_week' : 'Future_day_of_week',
    'is_weekend' : 'Future_is_weekend',
    'is_month_start' : 'Future_is_month_start',
    'is_month_end' : 'Future_is_month_end',
    'onpromotion' : 'Future_On_Promotion',
    'unit_sales' : 'target'
}
len(Future_Features_Map)

10

In [40]:
horizon = 28

In [41]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

transformer = FunctionTransformer(func=np.log1p, inverse_func=np.expm1)

In [42]:
from lightgbm import LGBMRegressor
import joblib
import polars as pl
from tqdm import trange

class RetailForecastingModel:
    def __init__(self, params, horizon):
        if params != None and len(params) != horizon:
            raise ValueError("the length of params should be the same as the horizon")
        
        self.horizon = horizon
        self.models = []
        for h in range(horizon):
            parameters = (params[h] if params != None else {}) | {
                'objective' : 'regression',
                'metric' : None,
                'device' : 'cpu',
                'n_jobs' : -1,
                'verbose' : -1
            }
            self.models.append(LGBMRegressor(**parameters))
    
    def fit(self, Dataset : pl.DataFrame, save_models = False, save_path = None):
        for h in trange(self.horizon, desc='Currently on Model'):
            Added_Future_Features_Dataset = Dataset.with_columns(
                [
                    pl.col(oldName).shift(-(h+1)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
                ]
            ).with_columns(
                [
                    pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
                ]
            ).with_columns(
                pl.col('item_nbr').shift(-self.horizon).over('item_nbr').alias('delete')
            )

            Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

            X = Added_Future_Features_Dataset.select(Features).to_pandas()
            y = Added_Future_Features_Dataset.get_column(Target).to_numpy()

            self.models[h].fit(X, y, categorical_feature=Categorical_Features)

            if save_models:
                joblib.dump(self.models[h], save_path+f'/Model{h+1}.pkl')
    
    def predict_sales(self, X : pl.DataFrame, transformer : FunctionTransformer):
        sales = []

        future_features_map = Future_Features_Map.copy()
        future_features_map.pop('unit_sales', None)

        for h in range(self.horizon):
            Added_Future_Features_X = X.with_columns(
                [
                    pl.col(oldName).shift(-(h+1)).over('item_nbr').alias(newName) for oldName, newName in future_features_map.items()
                ]
            ).with_columns(
                [
                    pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
                ]
            ).with_columns(
                pl.col('item_nbr').shift(-self.horizon).over('item_nbr').alias('delete')
            )

            Added_Future_Features_X = Added_Future_Features_X.drop_nulls()

            rounding_mask = Added_Future_Features_X.get_column('has_float_sales').to_numpy()

            Added_Future_Features_X = Added_Future_Features_X.select(Features).to_pandas()

            prediction = self.models[h].predict(Added_Future_Features_X)

            prediction = transformer.inverse_transform(prediction)

            prediction = np.clip(prediction, a_min=0, a_max=None)

            prediction = np.where(rounding_mask == 0, np.round(prediction), prediction)

            sales.append(prediction)

        return np.array(sales).T
    
    def load_models(self, models_path : str):
        for h in range(self.horizon):
            self.models[h] = joblib.load(models_path+f'/Model{h+1}.pkl')

In [43]:
def NWRMSLE_Evaluator(Dataset : pl.DataFrame, horizon : int, horizon_total : int, transformer : FunctionTransformer, y_pred_inverse_transformed = True):
    mapping = {
        0 : 1.0,
        1 : 1.5
    }
    Added_Future_Features_Dataset = Dataset.with_columns(
            [
                pl.col(oldName).shift(-(horizon)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
            ]
        ).with_columns(
            [
                pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
            ]
        ).with_columns(
            [
                pl.col('perishable').replace_strict(mapping, default=None).alias('weight'),
                pl.col('item_nbr').shift(-horizon_total).over('item_nbr').alias('delete')
            ]
        )

    Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

    rounding_mask = Added_Future_Features_Dataset.get_column('has_float_sales').to_numpy()

    X = Added_Future_Features_Dataset.select(Features).to_pandas()
    y = Added_Future_Features_Dataset.get_column(Target).to_numpy()
    weights = Added_Future_Features_Dataset.get_column('weight').to_numpy()
    
    def NWRMSLE(y_true, y_pred):
        if not y_pred_inverse_transformed:
            y_pred = transformer.inverse_transform(y_pred)
            y_pred = np.clip(y_pred, a_min=0, a_max=None)
            y_pred = np.where(rounding_mask == 0, np.round(y_pred), y_pred)
        y_pred_transformed = transformer.transform(y_pred)

        loss = (weights*(y_pred_transformed - y_true)**2).sum()/weights.sum()
        RE = loss / ((weights*y_true**2).sum()/weights.sum())

        return loss, RE
    
    return NWRMSLE, X, y

In [44]:
Train_set = pl.read_csv('./Store39_Dataset/MasterFullTrain.csv')
Test_set = pl.read_csv('./Store39_Dataset/MasterTest.csv')

In [45]:
params = []
for h in range(1, horizon+1):
    study = joblib.load(f'./optuna_studies/Optuna_Model{h}_Study.pkl')
    params.append(study.best_trial.params)

In [46]:
params = None

In [47]:
model = RetailForecastingModel(params=params, horizon=horizon)

In [48]:
model.load_models('./Models')

In [49]:
import os
models_path = f"./Models"
os.makedirs(models_path, exist_ok=True)

In [50]:
# model.fit(Dataset=Train_set, save_models=True, save_path=models_path)

In [51]:
predictions = model.predict_sales(Test_set, transformer)
predictions.shape

(6160, 28)

In [ ]:
errors = []
RLS = []
for h in range(horizon):
    evaluator, X_test, y_test = NWRMSLE_Evaluator(Test_set, h+1, horizon, transformer)
    error_RL = evaluator(y_test, predictions[:,h])
    errors.append(error_RL[0])
    RLS.append(error_RL[1]*100)
errors = np.array(errors)
RLS = np.array(RLS)
errors

array([0.38864051, 0.39565422, 0.39556583, 0.40050419, 0.40083767,
       0.42494658, 0.43162089, 0.44246669, 0.43846087, 0.4498414 ,
       0.46606564, 0.46151006, 0.47176771, 0.46287263, 0.46366002,
       0.44491084, 0.45748619, 0.47031912, 0.48051247, 0.47344284,
       0.46238324, 0.4488441 , 0.44924278, 0.44167043, 0.48679473,
       0.51206171, 0.48527296, 0.45638441])

In [53]:
RLS

array([0.18563491, 0.19384144, 0.17805378, 0.16940277, 0.15096743,
       0.16659651, 0.18476762, 0.21459694, 0.23610831, 0.22265932,
       0.20447089, 0.18056283, 0.18245531, 0.17208969, 0.17852025,
       0.19708095, 0.20278774, 0.18379945, 0.17797195, 0.19083927,
       0.1930042 , 0.19811111, 0.22003922, 0.22630575, 0.25231891,
       0.25551565, 0.22109336, 0.19599849])

In [54]:
errors.mean()

np.float64(0.44870502558309594)

In [55]:
RLS.mean()

np.float64(0.19769978829466142)